# 전체 투구 모델 — all_pitch_01 (초안)

- 목적: 초구 기록이 적은 투수에게 **모든 카운트 공**에서 배합 정보를 빌려 와 초구 구종 예측을 돕기 (블렌딩용)
  - 판단 기준은 이 모델 자체 성능이 아니라 **초구 모델(model_09)과 섞었을 때 2019 초구 log loss** (2020은 확인만)
  - 기준선: 수준 1 (model_08 0-3, README 4-15) = 모든 카운트 배합을 그대로 당기고 리그 "초구 / 모든 카운트" 비율을 곱함 -> model_07과 섞어 2019 교차 +3.8
- 이번 초안 = **수준 2: 투수 배합 + 리그 카운트 효과**
  - 수준 1의 문제: 2-2·0-2 같은 카운트 공의 배합이 그대로 섞임 -> 변화구를 결정구로 많이 쓰는 투수·카운트 분포가 다른 투수(불펜·선발)에서 0-0 배합이 어긋남
  - 수준 2: 공마다 "그 카운트·타자 손에서 리그가 던질 배합"(기대)과 실제를 비교해 투수의 **기대 대비 비율**(실제 / 기대, 구종별)을 누적 -> 0-0 리그 배합에 이 비율을 곱함
  - = 카운트 분포 차이를 걸러 낸 투수 성향 (간접 표준화)
- 정보 규칙: 구종은 트래킹 -> 전날까지만 (`fc.prior_table`·`fc.lookup`), 리그 카운트 표는 지난 시즌까지
- 노트북 구성: 1 데이터와 리그 카운트 효과 / 2 수준 1 재현 / 3 수준 2 (3-1 파라미터·블렌딩 비교, 3-2 진단) / 4 2020 확인 / 5 기록 수별 섞는 비율 / 6 저장 / 다음 할 일

In [1]:
%load_ext autoreload
%autoreload 2
import os, sys
if not os.path.exists("fp_common.py"):
    os.chdir("..")
sys.path.insert(0, os.getcwd())  # fp_common을 찾도록
import numpy as np
import pandas as pd
import fp_common as fc

MODEL = "all_pitch_01"

m, names = fc.load_data()
fp, y_type, y_zone, val, test = fc.first_pitches(m)
CLASSES, ZONES, MOM = fc.CLASSES, fc.ZONES, fc.MOM
q_val, q_test = fp[val], fp[test]
print(f"초구 {len(fp):,}개 (검증 {val.sum():,} / 평가 {test.sum():,})")
import lightgbm as lgb
from scipy.special import log_softmax
pd.set_option("display.max_rows", 200)
import lightgbm as lgb
from scipy.special import log_softmax
pd.set_option("display.max_rows", 200)

초구 50,709개 (검증 13,458 / 평가 12,615)


## 1. 데이터와 리그 카운트 효과

- 모든 카운트 공 = 구종 의심(`label_doubt`) 제외, 카운트 = `ball_before`-`strike_before` (12가지)
- 리그 카운트 표 `L[연도]`: 그 해 **전까지** 시즌의 카운트 × 타자 손(같은 손/반대 손)별 구종 비율 (분석 대상 41명 공)

In [2]:
M_ALL = m[m["label_doubt"].eq(False)].copy()  # 모든 카운트 공 (캐시가 같은 객체를 쓰도록 한 번만 만듦)
M_ALL[CLASSES] = (M_ALL["pitch_type"].astype(str).to_numpy()[:, None] == np.array(CLASSES)[None, :]).astype(float)
M_ALL["count"] = M_ALL["ball_before"].astype(int).astype(str) + "-" + M_ALL["strike_before"].astype(int).astype(str)
COUNTS = sorted(M_ALL["count"].unique())
print(f"모든 카운트 공 {len(M_ALL):,}개 (초구 {M_ALL['count'].eq('0-0').sum():,}개, 비율 {M_ALL['count'].eq('0-0').mean():.2f})")

def league_count_table(year):  # 그 해 전까지 시즌: (카운트, 같은 손) -> 구종 비율 9개
    past = M_ALL[M_ALL["GYEAR"] < year]
    return past.groupby(["count", "same_side"])[CLASSES].mean()

L = {yr: league_count_table(yr) for yr in (fc.VAL_YEAR, fc.TEST_YEAR)}
# 카운트 효과 보기 (2017–18, 손 합침)
past = M_ALL[M_ALL["GYEAR"] < fc.VAL_YEAR]
cnt_tab = past.groupby("count")[CLASSES].mean()
cnt_tab.insert(0, "공 수", past.groupby("count").size())
display(cnt_tab[["공 수", "FAST", "TWOS", "SLID", "CURV", "CHUP", "FORK", "CUTT"]].round(3))

모든 카운트 공 197,097개 (초구 50,709개, 비율 0.26)


,공 수,FAST,TWOS,SLID,CURV,CHUP,FORK,CUTT
count,,,,,,,,
0-0,24636,0.506,0.055,0.157,0.099,0.129,0.051,0.002
0-1,12082,0.448,0.041,0.152,0.145,0.128,0.082,0.003
0-2,5933,0.479,0.037,0.139,0.091,0.130,0.122,0.002
1-0,9728,0.522,0.054,0.164,0.060,0.142,0.056,0.002
1-1,10044,0.445,0.038,0.174,0.108,0.142,0.089,0.003
1-2,9613,0.434,0.026,0.161,0.105,0.135,0.135,0.002
2-0,3236,0.658,0.057,0.112,0.023,0.119,0.029,0.002
2-1,5081,0.545,0.045,0.160,0.044,0.143,0.059,0.002
2-2,7949,0.456,0.027,0.157,0.091,0.146,0.119,0.003


## 2. 수준 1 재현 (model_08 0-3과 같은 구조)

- 투수 -> 투수×타자 손 모든 카운트 배합을 당김 (alpha 7.5·hand_k 5.48 = B4 값), 기준점 = 지난 시즌 리그 모든 카운트 배합
- 0-0으로 맞춤: 리그 "초구 배합 / 모든 카운트 배합" 비율을 곱함
- 2019 교차에서 고른 값: 반감 100구, 시즌 초 배수 0.25

In [3]:
def level1_00(q, year, half_life=100.0, season_f=0.25, alpha=7.5, hand_k=5.48):  # 수준 1 0-0 구종 확률 (n, 9)
    past = M_ALL[M_ALL["GYEAR"] < year]
    ref_all = past[CLASSES].mean().to_numpy()
    ref_00 = fp.loc[fp["GYEAR"] < year, CLASSES].mean().to_numpy()
    out = []
    for keys in (["pitcher_code"], ["pitcher_code", "same_side"]):
        S, N = fc.lookup(fc.prior_table(M_ALL, keys, CLASSES, half_life, season_f=season_f), q, keys, CLASSES, season_f=season_f)
        out += [S, N[:, :1]]
    S, N, S2, N2 = out
    P = fc.shrink(S2, N2, fc.shrink(S, N, ref_all, alpha), hand_k) * (ref_00 / np.maximum(ref_all, 1e-6))
    return P / P.sum(1, keepdims=True)

Q = {fc.VAL_YEAR: fp[val], fc.TEST_YEAR: fp[test]}
Y = {fc.VAL_YEAR: y_type[val], fc.TEST_YEAR: y_type[test]}
nll = lambda P, y: -np.log(np.clip(P[np.arange(len(y)), y], 1e-12, None))
P_L1 = {yr: level1_00(q, yr) for yr, q in Q.items()}
P13 = {yr: fc.align(pd.read_parquet("results/model_09/per_pitch.parquet"), fp, yr)[fc.P_TYPE].to_numpy() for yr in Q}  # 초구 모델 (2단계 최종)
print(f"2019 초구 log loss: 수준 1만 {nll(P_L1[fc.VAL_YEAR], Y[fc.VAL_YEAR]).mean():.4f} (model_08 0-3 1.0973) / model_09 {nll(P13[fc.VAL_YEAR], Y[fc.VAL_YEAR]).mean():.4f}")

2019 초구 log loss: 수준 1만 1.0973 (model_08 0-3 1.0973) / model_09 1.0613


## 3. 수준 2: 투수 배합 + 리그 카운트 효과

- 공마다 기대 배합 e = `L[연도]`(그 공의 카운트, 타자 손) -> 투수(×손)별로 실제 O = Σ 원-핫, 기대 E = Σ e를 전날까지 누적 (반감기·시즌 초 배수)
- 기대 대비 비율 (구종별, 표본 적으면 1 쪽 또는 상위 그룹 쪽으로 당김)
  - 투수: r1 = (O1 + k1·ē) / (E1 + k1·ē), ē = 리그 모든 카운트 배합
  - 투수×손: r2 = (O2 + k2·r1·ē) / (E2 + k2·ē) -> 기록 없으면 r1, r1도 없으면 1
- 0-0 예측: P ∝ `L[연도]`(0-0, 타자 손) × r2 / 다른 카운트 예측: P ∝ L(그 카운트, 손) × r2 (아래 자체 점수용)
- k1·k2는 **모든 카운트 공 수** 단위 (B4 alpha 7.5는 초구 수 단위라 그대로 못 씀) -> 격자로
- 연도마다 기대 열이 달라짐 (`L[연도]` 기준) -> 연도별 데이터 사본(`M_E[연도]`)을 한 번 만들어 캐시가 같은 객체를 쓰게

In [4]:
E_COLS = ["e_" + c for c in CLASSES]
M_E = {}
for yr in Q:
    d = M_ALL[["gameID", "pit_no", "game_date", "GYEAR", "pitcher_code", "same_side", "count"] + CLASSES].copy()
    e = L[yr].reindex(pd.MultiIndex.from_frame(d[["count", "same_side"]])).to_numpy()
    d[E_COLS] = e
    M_E[yr] = d[d["GYEAR"] < yr + 1]  # 그 해까지 (조회는 lookup이 전날까지로 자름)

def level2(q, year, half_life, season_f, k1, k2, count=None, data=None):  # 수준 2 구종 확률 (n, 9), count=None이면 0-0
    d = M_E[year] if data is None else data  # data: 검사용 임시 데이터 (캐시 안 씀)
    ebar = d.loc[d["GYEAR"] < year, CLASSES].mean().to_numpy()  # 리그 모든 카운트 배합 (지난 시즌까지)
    R = []
    for keys in (["pitcher_code"], ["pitcher_code", "same_side"]):
        t = fc.prior_table(d, keys, CLASSES + E_COLS, half_life, season_f=season_f, cache=data is None)
        S, _ = fc.lookup(t, q, keys, CLASSES + E_COLS, season_f=season_f)
        R.append((S[:, :9], S[:, 9:]))
    (O1, E1), (O2, E2) = R
    r1 = (O1 + k1 * ebar) / (E1 + k1 * ebar)
    r2 = (O2 + k2 * r1 * ebar) / (E2 + k2 * ebar)
    cnt = np.full(len(q), "0-0") if count is None else count
    base = L[year].reindex(pd.MultiIndex.from_arrays([cnt, q["same_side"].to_numpy()])).to_numpy()
    P = base * r2
    return P / P.sum(1, keepdims=True)

# 미래 정보 확인: 2019 첫 경기일 공의 r은 2018까지 기록만 -> 같은 날 기록을 바꿔도 예측이 같아야 함
q0 = Q[fc.VAL_YEAR].iloc[:200]
P_a = level2(q0, fc.VAL_YEAR, 100.0, 0.25, 30.0, 30.0)
first_day = q0["game_date"].min()
tmp = M_E[fc.VAL_YEAR].copy()
tmp.loc[tmp["game_date"] >= first_day, CLASSES] = 1.0 / 9  # 그날 이후 구종을 지움
P_b = level2(q0, fc.VAL_YEAR, 100.0, 0.25, 30.0, 30.0, data=tmp)
same_day = q0["game_date"].eq(first_day).to_numpy()
assert np.allclose(P_a[same_day], P_b[same_day])
tmp.loc[tmp["game_date"] < first_day, CLASSES] = 1.0 / 9  # 대조: 그 전 구종을 지우면 달라져야 함
assert not np.allclose(P_a[same_day], level2(q0, fc.VAL_YEAR, 100.0, 0.25, 30.0, 30.0, data=tmp)[same_day])
print(f"미래 정보 확인 ({same_day.sum()}개): 그날 이후 구종을 지워도 예측 같음, 그 전 구종을 지우면 달라짐")
del tmp

미래 정보 확인 (40개): 그날 이후 구종을 지워도 예측 같음, 그 전 구종을 지우면 달라짐


### 3-1. 수준 2 파라미터 (2019, 블렌딩 기준)

- 격자: 반감기(모든 카운트 공 수) 12.5·25·50·100·200 × k1 1·3·10·30·100 × k2 1·3·10·30·100, 시즌 초 배수 0.25 (수준 1에서 고른 값)
  - 처음 격자(반감 50–200, k 10–100)에서 1등이 모두 아래 끝 -> 아래로 넓힘, 넓힌 뒤에도 끝이면 표시
- 각 후보를 model_09과 섞음 P = (1 − w)·P_model_09 + w·P_전체, w 0–0.5
- 고르기: 2019 경기 단위 5겹 교차 (다른 겹에서 고른 후보·w로 이 겹 채점) -> 2019 정직한 이득, 수준 1도 같은 방법으로 (기준선을 model_09 위에서 다시 잼)

In [5]:
q19, y19 = Q[fc.VAL_YEAR], Y[fc.VAL_YEAR]
g19 = q19["gameID"].to_numpy()
W = np.round(np.arange(0, 0.51, 0.05), 2)
HLS, KS = (12.5, 25.0, 50.0, 100.0, 200.0), (1.0, 3.0, 10.0, 30.0, 100.0)
GRID = [(hl, k1, k2) for hl in HLS for k1 in KS for k2 in KS]
CAND2 = {g: level2(q19, fc.VAL_YEAR, g[0], 0.25, g[1], g[2]) for g in GRID}
base19 = nll(P13[fc.VAL_YEAR], y19)
ug = np.unique(g19)
FOLD = pd.Series(np.random.default_rng(0).integers(0, 5, len(ug)), index=ug).reindex(g19).to_numpy()

def game_ci(d, gid, B=1000):  # 경기 단위 부트스트랩 95% 구간
    s = pd.DataFrame({"g": gid, "d": d}).groupby("g")["d"].agg(["sum", "count"])
    i = np.random.default_rng(0).integers(0, len(s), (B, len(s)))
    return np.percentile(s["sum"].to_numpy()[i].sum(1) / s["count"].to_numpy()[i].sum(1), [2.5, 97.5])

def cv_blend(cands):  # 후보 {이름: P} -> 5겹 교차 공별 log loss, 겹마다 고른 (후보, w), 전체 2019로 고른 (후보, w)
    LLC = {(k, w): nll((1 - w) * P13[fc.VAL_YEAR] + w * P, y19) for k, P in cands.items() for w in W}
    cv, picked = np.empty(len(y19)), []
    for f in range(5):
        best = min(LLC, key=lambda kw: LLC[kw][FOLD != f].mean())
        picked.append(best)
        cv[FOLD == f] = LLC[best][FOLD == f]
    return cv, picked, min(LLC, key=lambda kw: LLC[kw].mean()), LLC

rows, CV = [], {}
for nm, cands in {"수준 1": {"반감 100": P_L1[fc.VAL_YEAR]}, "수준 2": CAND2}.items():
    cv, picked, best, LLC = cv_blend(cands)
    CV[nm] = (cv, best, LLC)
    d = base19 - cv
    lo, hi = game_ci(d, g19)
    rows.append({"전체 투구 모델": nm, "단독 2019 log loss": min(nll(P, y19).mean() for P in cands.values()),
                 "교차 블렌딩 이득 ×1000": d.mean() * 1000, "95% 하한": lo * 1000, "95% 상한": hi * 1000,
                 "2019 전체로 고른 값": best, "겹마다 고른 값": picked})
display(pd.DataFrame(rows).set_index("전체 투구 모델").round(4))
d12 = CV["수준 1"][0] - CV["수준 2"][0]  # 양수 = 수준 2가 좋음
lo, hi = game_ci(d12, g19)
print(f"수준 2 − 수준 1 (교차 블렌딩, ×1000): {d12.mean() * 1000:.2f} (95% {lo * 1000:.2f} ~ {hi * 1000:.2f})")
tab = pd.DataFrame([{"반감기": k[0][0], "k1": k[0][1], "k2": k[0][2], "w": k[1], "2019 log loss": v.mean()}
                    for k, v in CV["수준 2"][2].items()]).sort_values("2019 log loss")
display(tab.head(10).round(5))
(hl_b, k1_b, k2_b), _ = CV["수준 2"][1]
print("1등이 격자 끝:", {n: v for n, v, g in [("반감기", hl_b, HLS), ("k1", k1_b, KS), ("k2", k2_b, KS)] if v in (g[0], g[-1])} or "없음")

,단독 2019 log loss,교차 블렌딩 이득 ×1000,95% 하한,95% 상한,2019 전체로 고른 값,겹마다 고른 값
전체 투구 모델,,,,,,
수준 1,1.0973,3.8310,1.7090,5.8430,"(반감 100, 0.2)","[(반감 100, 0.2), (반감 100, 0.2), (반감 100, 0.2), ..."
수준 2,1.0943,3.8999,1.1243,6.5831,"((25.0, 1.0, 3.0), 0.2)","[((25.0, 10.0, 1.0), 0.2), ((25.0, 1.0, 3.0), ..."


수준 2 − 수준 1 (교차 블렌딩, ×1000): 0.07 (95% -1.28 ~ 1.36)


,반감기,k1,k2,w,2019 log loss
290,25.0,1.0,3.0,0.20,1.05614
279,25.0,1.0,1.0,0.20,1.05614
334,25.0,3.0,1.0,0.20,1.05615
345,25.0,3.0,3.0,0.20,1.05616
389,25.0,10.0,1.0,0.20,1.05617
301,25.0,1.0,10.0,0.20,1.05622
444,25.0,30.0,1.0,0.20,1.05624
400,25.0,10.0,3.0,0.20,1.05628
356,25.0,3.0,10.0,0.20,1.05633
291,25.0,1.0,3.0,0.25,1.05633


1등이 격자 끝: {'k1': 1.0}


### 3-2. 진단 (2019)

- 기록 수별 이득: 전날까지 그 투수 초구 수 구간별 교차 블렌딩 이득 (수준 1·2)
- 전체 투구 모델 자체 점수: 2019 **모든 카운트 공** log loss = 리그 카운트 표만 / 수준 2 (고른 값) -> 카운트 효과·투수 비율이 제대로 작동하는지

In [6]:
_, N_raw = fc.lookup(fc.prior_table(fp, ["pitcher_code"], CLASSES[:1]), fp, ["pitcher_code"], CLASSES[:1])
nb = pd.cut(N_raw[val, 0], [-1, 50, 200, 500, 1e9], labels=["0–50", "51–200", "201–500", "501+"]).to_numpy()
display(pd.DataFrame({"초구": pd.Series(nb).value_counts(sort=False),
                      **{f"{nm} 이득 ×1000": pd.Series(base19 - CV[nm][0]).groupby(nb, observed=True).mean() * 1000 for nm in CV}})
        .rename_axis("전날까지 투수 초구 수").round(2))

(hl, k1, k2), w_best = CV["수준 2"][1]
a19 = M_ALL[M_ALL["GYEAR"].eq(fc.VAL_YEAR)]
ya = a19["pitch_type"].map(fc.TYPE_IDX).to_numpy()
P_cnt = L[fc.VAL_YEAR].reindex(pd.MultiIndex.from_frame(a19[["count", "same_side"]])).to_numpy()
P_l2a = level2(a19, fc.VAL_YEAR, hl, 0.25, k1, k2, count=a19["count"].to_numpy())
a_rows = {"리그 카운트 표만": nll(P_cnt, ya), f"수준 2 (반감 {hl:g}, k1 {k1:g}, k2 {k2:g})": nll(P_l2a, ya)}
by_cnt = pd.DataFrame({nm: pd.Series(v).groupby(a19["count"].to_numpy()).mean() for nm, v in a_rows.items()})
by_cnt.loc["전체"] = [v.mean() for v in a_rows.values()]
display(by_cnt.round(4).rename_axis("2019 모든 카운트 log loss"))

,초구,수준 1 이득 ×1000,수준 2 이득 ×1000
전날까지 투수 초구 수,,,
0–50,211,24.17,36.86
201–500,3405,5.43,5.96
501+,9203,2.32,1.54
51–200,639,10.32,16.08


,리그 카운트 표만,"수준 2 (반감 25, k1 1, k2 3)"
2019 모든 카운트 log loss,,
0-0,1.4342,1.1146
0-1,1.5557,1.2495
0-2,1.5294,1.2016
1-0,1.4164,1.0549
1-1,1.5501,1.1884
1-2,1.5510,1.2048
2-0,1.1924,0.8696
2-1,1.3461,0.9675
2-2,1.5084,1.1283


## 4. 2020 확인

- 3-1에서 2019 전체로 고른 것 하나만 (수준 1·2 중 교차 이득이 큰 쪽, 후보·w 그대로)
- model_09 대비 2020 초구 log loss, 경기 단위 부트스트랩

In [7]:
pick = max(CV, key=lambda nm: (base19 - CV[nm][0]).mean())  # 2019 교차 이득이 큰 쪽
key, w = CV[pick][1]
P20_all = P_L1[fc.TEST_YEAR] if pick == "수준 1" else level2(Q[fc.TEST_YEAR], fc.TEST_YEAR, key[0], 0.25, key[1], key[2])
y20 = Y[fc.TEST_YEAR]
P20_blend = (1 - w) * P13[fc.TEST_YEAR] + w * P20_all
d = nll(P13[fc.TEST_YEAR], y20) - nll(P20_blend, y20)
lo, hi = game_ci(d, Q[fc.TEST_YEAR]["gameID"].to_numpy())
print(f"고른 것: {pick} {key}, w {w}")
print(f"2020 초구 log loss: model_09 {nll(P13[fc.TEST_YEAR], y20).mean():.4f} -> 블렌딩 {nll(P20_blend, y20).mean():.4f}, "
      f"이득 ×1000 {d.mean() * 1000:.2f} (95% {lo * 1000:.2f} ~ {hi * 1000:.2f})")

고른 것: 수준 2 (25.0, 1.0, 3.0), w 0.2
2020 초구 log loss: model_09 1.0561 -> 블렌딩 1.0523, 이득 ×1000 3.77 (95% 1.74 ~ 5.87)


#### 🔍 결과
#### 수준 2는 단독으로는 수준 1보다 좋지만, model_09과 섞은 이득은 수준 1과 거의 같음 (2019 교차 +3.9 vs +3.8)
- 단독 2019 초구 log loss: 수준 1 1.0973 -> 수준 2 1.0943 (B4 1.0937과 비슷) -> 카운트 분포를 걸러 낸 효과는 있음
- 교차 블렌딩 이득 ×1000 (model_09 위): 수준 1 +3.8 (95% +1.7 ~ +5.8), 수준 2 +3.9 (+1.1 ~ +6.6), 차이 +0.1 (−1.3 ~ +1.4) -> 차이 없음
  - 처음 격자(반감 50–200, k 10–100)에서는 +4.6이었지만 1등이 모두 아래 끝 -> 넓히니 후보가 많아져 겹마다 고른 값이 흔들리고 교차 이득이 내려옴 (고르는 몫의 잡음)
- 파라미터: 반감 25구가 뚜렷이 좋고 (12.5·50보다), k1·k2는 1–30 사이 거의 평평 (1.05614–1.05633) -> 평평한 구간 규칙이면 덜 극단적인 값(k 10 근처)으로, k1 1등이 아래 끝이지만 평평해서 더 넓힐 의미 작음
  - w는 0.2 (model_09 80% + 전체 투구 20%)로 수준 1과 같음
- 기록 수별: 초구 0–50개 투수 +24 -> +37, 51–200개 +10 -> +16로 수준 2가 큼 / 501개+ +2.3 -> +1.5로 작음 -> **기록 적은 투수에서는 수준 2가 낫고 많은 투수에서는 수준 1이 나음** -> 기록 수에 따라 섞는 비율을 다르게 하면 둘의 장점을 같이 쓸 수 있음
- 자체 점수 (2019 모든 카운트): 리그 카운트 표만 1.446 -> 수준 2 1.109, 반감 50이면 1.092 -> 모든 카운트 예측에는 더 긴 반감기가 맞고, 초구 블렌딩에는 짧은 반감기(최근 배합)가 맞음
- 2020 확인 (수준 2 반감 25·k1 1·k2 3, w 0.2): 1.0561 -> 1.0523, 이득 +3.8 (95% +1.7 ~ +5.9) -> 2019 크기 그대로 재현
- 정리: 전체 투구 블렌딩 자체는 두 해 모두 약 −0.004로 안정적 (2단계 model_06 시즌 초 배수와 비슷한 크기), 수준 1 -> 2 고도화의 몫은 기록 적은 투수에 한정


## 5. 기록 수별 섞는 비율

- 4장 결과: 기록 적은 투수에서는 수준 2 이득이 크고, 기록 많은 투수에서는 작음 -> 섞는 비율 w를 투수 기록 수 n에 따라
  - w(n) = w∞ + (w0 − w∞) · c / (n + c): 기록 없을 때 w0, 많아지면 w∞ 쪽으로, c = 절반쯤 넘어가는 기록 수
- n = **B4 투수 유효 초구 수** (반감 35.4구·시즌 초 0.25로 센 값, 최대 약 52) -> 해마다 범위가 같음
  - 그냥 누적 초구 수는 해마다 커짐 (`bat_n` 문제, model_07 0-2) -> 2019에서 고른 c가 2020에 안 맞음
- 전체 투구 모델 값은 고정: 수준 2 = 반감 25·k1 10·k2 10 (3-1에서 k는 평평 -> 덜 극단적인 값), 수준 1 = 반감 100 -> 고르는 것은 w0·w∞·c만
- 비교: 고정 w (4장과 같은 방식) / 수준 1 w(n) / 수준 2 w(n) / 수준 1·2 둘 다 섞기 (각자 w(n)), 모두 2019 경기 단위 5겹 교차, 2020은 1등만 확인

In [8]:
_, N_b4 = fc.lookup(fc.prior_table(fp, ["pitcher_code"], CLASSES, 35.4, season_f=0.25), fp, ["pitcher_code"], CLASSES, season_f=0.25)
NB = {fc.VAL_YEAR: N_b4[val, 0], fc.TEST_YEAR: N_b4[test, 0]}  # B4 투수 유효 초구 수 (전날까지)
print("B4 유효 초구 수 (2019 / 2020):", {yr: np.percentile(v, [10, 50, 90]).round(1).tolist() for yr, v in NB.items()})
PA = {"수준 1": {yr: P_L1[yr] for yr in Q}, "수준 2": {yr: level2(Q[yr], yr, 25.0, 0.25, 10.0, 10.0) for yr in Q}}

W0, WINF, C = (0.2, 0.3, 0.4, 0.5, 0.6, 0.8), (0.0, 0.05, 0.1, 0.15, 0.2, 0.3), (2.0, 5.0, 10.0, 20.0, 40.0)
def wfun(n, w0, winf, c):
    return winf + (w0 - winf) * c / (n + c)

def blend(yr, parts):  # parts: [(이름, (w0, w∞, c))] -> 섞은 확률
    n, P = NB[yr], P13[yr].copy()
    ws = [wfun(n, *p) for _, p in parts]
    P = P13[yr] * (1 - sum(ws))[:, None]
    for (nm, _), w in zip(parts, ws):
        P = P + w[:, None] * PA[nm][yr]
    return P

def cv_pick(cands):  # {키: 2019 공별 log loss} -> 5겹 교차 공별 log loss, 2019 전체 1등
    cv = np.empty(len(y19))
    for f in range(5):
        best = min(cands, key=lambda k: cands[k][FOLD != f].mean())
        cv[FOLD == f] = cands[best][FOLD == f]
    return cv, min(cands, key=lambda k: cands[k].mean())

SETS = {
    "고정 w 수준 1": {(("수준 1", (w, w, 1.0)),): None for w in W},
    "고정 w 수준 2": {(("수준 2", (w, w, 1.0)),): None for w in W},
    "w(n) 수준 1": {(("수준 1", (a, b, c)),): None for a in W0 for b in WINF for c in C if b <= a},
    "w(n) 수준 2": {(("수준 2", (a, b, c)),): None for a in W0 for b in WINF for c in C if b <= a},
    "w(n) 수준 1 + 수준 2": {(("수준 1", (b1, b1, 1.0)), ("수준 2", (a2, 0.0, c))): None
                             for b1 in (0.0, 0.1, 0.15, 0.2) for a2 in (0.2, 0.3, 0.4, 0.6) for c in C},  # 수준 1 고정 w + 수준 2는 기록 적을 때만
}
RES, rows = {}, []
for nm, ks in SETS.items():
    LL = {k: nll(blend(fc.VAL_YEAR, list(k)), y19) for k in ks}
    cv, best = cv_pick(LL)
    RES[nm] = (cv, best)
    d = base19 - cv
    lo, hi = game_ci(d, g19)
    rows.append({"방법": nm, "후보 수": len(ks), "교차 이득 ×1000": d.mean() * 1000, "95% 하한": lo * 1000, "95% 상한": hi * 1000,
                 "2019 전체 1등": "; ".join(f"{p[0]} w0 {p[1][0]:g} w∞ {p[1][1]:g} c {p[1][2]:g}" for p in best)})
display(pd.DataFrame(rows).set_index("방법").round(3))
ref = RES["고정 w 수준 1"][0]
for nm in RES:
    if nm != "고정 w 수준 1":
        d = ref - RES[nm][0]
        lo, hi = game_ci(d, g19)
        print(f"{nm} − 고정 w 수준 1 (×1000): {d.mean() * 1000:+.2f} (95% {lo * 1000:+.2f} ~ {hi * 1000:+.2f})")
bins = pd.cut(NB[fc.VAL_YEAR], [-0.1, 5, 20, 40, 100], labels=["0–5", "5–20", "20–40", "40+"]).to_numpy()
display(pd.DataFrame({"초구": pd.Series(bins).value_counts(sort=False),
                      **{nm: pd.Series(base19 - RES[nm][0]).groupby(bins, observed=True).mean() * 1000 for nm in RES}})
        .rename_axis("B4 유효 초구 수 (이득 ×1000)").round(2))

B4 유효 초구 수 (2019 / 2020): {2019: [28.2, 50.2, 51.6], 2020: [27.8, 49.9, 51.6]}


,후보 수,교차 이득 ×1000,95% 하한,95% 상한,2019 전체 1등
방법,,,,,
고정 w 수준 1,11,3.831,1.709,5.843,수준 1 w0 0.2 w∞ 0.2 c 1
고정 w 수준 2,11,4.447,1.987,7.018,수준 2 w0 0.2 w∞ 0.2 c 1
w(n) 수준 1,175,3.676,1.516,5.799,수준 1 w0 0.4 w∞ 0 c 40
w(n) 수준 2,175,4.878,2.366,7.329,수준 2 w0 0.6 w∞ 0 c 20
w(n) 수준 1 + 수준 2,80,4.172,1.701,6.542,수준 1 w0 0 w∞ 0 c 1; 수준 2 w0 0.6 w∞ 0 c 20


고정 w 수준 2 − 고정 w 수준 1 (×1000): +0.62 (95% -0.78 ~ +2.07)
w(n) 수준 1 − 고정 w 수준 1 (×1000): -0.15 (95% -0.39 ~ +0.09)
w(n) 수준 2 − 고정 w 수준 1 (×1000): +1.05 (95% -0.39 ~ +2.57)
w(n) 수준 1 + 수준 2 − 고정 w 수준 1 (×1000): +0.34 (95% -0.85 ~ +1.59)


,초구,고정 w 수준 1,고정 w 수준 2,w(n) 수준 1,w(n) 수준 2,w(n) 수준 1 + 수준 2
B4 유효 초구 수 (이득 ×1000),,,,,,
0–5,40,-4.47,16.16,-10.70,36.94,38.05
20–40,1934,9.03,9.96,9.52,10.90,10.44
40+,10780,2.95,2.90,2.85,3.17,2.62
5–20,704,3.53,12.32,1.14,12.64,8.84


In [9]:
# 2020 확인: 2019 교차 이득 1등 방법의 2019 전체 1등 값 + 비교용 고정 w 수준 1
top = max(RES, key=lambda nm: (base19 - RES[nm][0]).mean())
y20, g20 = Y[fc.TEST_YEAR], Q[fc.TEST_YEAR]["gameID"].to_numpy()
b20 = nll(P13[fc.TEST_YEAR], y20)
rows = []
for nm in dict.fromkeys([top, "고정 w 수준 1"]):
    d = b20 - nll(blend(fc.TEST_YEAR, list(RES[nm][1])), y20)
    lo, hi = game_ci(d, g20)
    rows.append({"방법": nm, "2020 이득 ×1000 (model_09 대비)": d.mean() * 1000, "95% 하한": lo * 1000, "95% 상한": hi * 1000})
display(pd.DataFrame(rows).set_index("방법").round(3))
if top != "고정 w 수준 1":
    d = nll(blend(fc.TEST_YEAR, list(RES["고정 w 수준 1"][1])), y20) - nll(blend(fc.TEST_YEAR, list(RES[top][1])), y20)
    lo, hi = game_ci(d, g20)
    print(f"2020 {top} − 고정 w 수준 1 (×1000): {d.mean() * 1000:+.2f} (95% {lo * 1000:+.2f} ~ {hi * 1000:+.2f})")

,2020 이득 ×1000 (model_09 대비),95% 하한,95% 상한
방법,,,
w(n) 수준 2,3.693,1.675,5.980
고정 w 수준 1,2.910,1.110,4.656


2020 w(n) 수준 2 − 고정 w 수준 1 (×1000): +0.78 (95% -0.63 ~ +2.61)


### 5-1. 비율 효과만 따로, 반감기도 교차 안에서 고르기 (사용자 요청)

- 위 표의 "w(n) 수준 2 − 고정 w 수준 1"에는 수준 효과와 비율 효과가 섞여 있음 -> 나눠서 봄
  - ① 비율 효과 = 고정 w 수준 2 vs w(n) 수준 2 (둘 다 반감 25 고정, 같은 조건)
- 반감 25는 2019 전체로 고른 값 -> 위 교차 이득이 조금 부풀려짐
  - ② 정직한 크기: 두 방법 모두 반감기(12.5·25·50)도 겹마다 같이 고름 (k 10 고정), 두 방법이 같은 방식이라 ①의 비교도 유지
- 2020: 각 방법의 2019 전체 1등 값으로 확인

In [10]:
HL5 = (12.5, 25.0, 50.0)
PH = {hl: {yr: level2(Q[yr], yr, hl, 0.25, 10.0, 10.0) for yr in Q} for hl in HL5}  # 수준 2, 반감기별

def blend_h(yr, hl, w0, winf, c):  # 수준 2(반감 hl)를 w(n)으로 섞음
    w = wfun(NB[yr], w0, winf, c)[:, None]
    return (1 - w) * P13[yr] + w * PH[hl][yr]

SET2 = {"고정 w": [(w, w, 1.0) for w in W], "w(n)": [(a, b, cc) for a in W0 for b in WINF for cc in C if b <= a]}
rows, OUT = [], {}
for mode, hls in {"반감 25 고정": (25.0,), "반감기도 교차 안에서": HL5}.items():
    for nm, ps in SET2.items():
        LL = {(hl, p): nll(blend_h(fc.VAL_YEAR, hl, *p), y19) for hl in hls for p in ps}
        cv, best = cv_pick(LL)
        d20 = b20 - nll(blend_h(fc.TEST_YEAR, best[0], *best[1]), y20)
        OUT[(mode, nm)] = (cv, d20)
        lo, hi = game_ci(base19 - cv, g19)
        lo2, hi2 = game_ci(d20, g20)
        rows.append({"반감기": mode, "섞는 비율": nm, "후보 수": len(LL), "2019 교차 이득 ×1000": (base19 - cv).mean() * 1000,
                     "95% 구간": f"{lo * 1000:+.1f} ~ {hi * 1000:+.1f}", "2020 이득 ×1000": d20.mean() * 1000,
                     "2020 95% 구간": f"{lo2 * 1000:+.1f} ~ {hi2 * 1000:+.1f}",
                     "2019 전체 1등": f"반감 {best[0]:g}, w0 {best[1][0]:g} w∞ {best[1][1]:g} c {best[1][2]:g}"})
display(pd.DataFrame(rows).set_index(["반감기", "섞는 비율"]).round(2))
rows = []
for mode in ("반감 25 고정", "반감기도 교차 안에서"):  # ① 비율 효과 = 고정 w -> w(n)
    (cv_f, d20_f), (cv_n, d20_n) = OUT[(mode, "고정 w")], OUT[(mode, "w(n)")]
    d19, d20 = cv_f - cv_n, d20_n - d20_f  # 양수 = w(n)이 좋음
    lo, hi = game_ci(d19, g19)
    lo2, hi2 = game_ci(d20, g20)
    rows.append({"반감기": mode, "비율 효과 2019 ×1000": d19.mean() * 1000, "2019 95% 구간": f"{lo * 1000:+.2f} ~ {hi * 1000:+.2f}",
                 "비율 효과 2020 ×1000": d20.mean() * 1000, "2020 95% 구간": f"{lo2 * 1000:+.2f} ~ {hi2 * 1000:+.2f}"})
display(pd.DataFrame(rows).set_index("반감기").round(2))

후보 수  2019 교차 이득 ×1000       95% 구간  2020 이득 ×1000  2020 95% 구간                  2019 전체 1등
반감기         섞는 비율                                                                                             
반감 25 고정    고정 w     11              4.45  +2.0 ~ +7.0           3.55  +1.5 ~ +5.7    반감 25, w0 0.2 w∞ 0.2 c 1
            w(n)    175              4.88  +2.4 ~ +7.3           3.69  +1.7 ~ +6.0     반감 25, w0 0.6 w∞ 0 c 20
반감기도 교차 안에서 고정 w     33              4.22  +1.8 ~ +6.7           3.67  +1.8 ~ +5.5    반감 50, w0 0.2 w∞ 0.2 c 1
            w(n)    525              4.26  +1.7 ~ +6.6           3.62  +1.6 ~ +5.6  반감 50, w0 0.6 w∞ 0.05 c 20

,비율 효과 2019 ×1000,2019 95% 구간,비율 효과 2020 ×1000,2020 95% 구간
반감기,,,,
반감 25 고정,0.43,-0.06 ~ +0.94,0.15,-0.34 ~ +0.65
반감기도 교차 안에서,0.04,-0.53 ~ +0.67,-0.05,-0.38 ~ +0.25


#### 🔍 5장 결과
#### 기록 수별 섞는 비율의 효과는 없음 (반감기도 교차 안에서 고르면 2019 +0.04, 2020 −0.05) -> 고정 비율로 충분, 블렌딩 설정 = 수준 2 반감 50·k 10 + 고정 w 0.2
- 처음 표 (반감 25 고정): w(n) 수준 2 +4.9로 1등, 고정 w 수준 1보다 +1.0 -> 그런데 이 차이에는 수준 효과와 비율 효과가 섞여 있음 (5-1에서 나눔)
- 5-1 비율 효과만 (고정 w 수준 2 -> w(n) 수준 2, 같은 조건)
  - 반감 25 고정: 2019 +0.43 (95% −0.06 ~ +0.94), 2020 +0.15 (−0.34 ~ +0.65)
  - **반감기도 교차 안에서: 2019 +0.04 (−0.53 ~ +0.67), 2020 −0.05 (−0.38 ~ +0.25)** -> 사실상 0
  - 기록 적은 투수(B4 유효 초구 0–20개)에서 크게 보였던 차이는 공 약 750개(6%)뿐이고, 고정 w 수준 2도 그 투수들에서 이미 많이 얻음 (0–5개 +16, 5–20개 +12)
- 반감 25 고정이 정확하지 않았음: 반감 25는 3-1에서 k 1–3과 함께 1등이었고, k 10으로 고정하면 반감 50이 더 좋음 (5-1 2019 전체 1등이 두 방법 모두 반감 50) -> k를 평평하다고 따로 정한 것이 반감기와 연결돼 있었음 (연결된 파라미터는 같이 고르는 규칙)
- 정직한 블렌딩 크기 (반감기도 교차 안에서): 고정 w +4.2 (95% +1.8 ~ +6.7), 2020 +3.7 (+1.8 ~ +5.5)
- 수준 1 (+3.8) -> 수준 2 (+4.2)는 +0.4 정도, 95% 구간에 0 포함 (3-1 +0.1, 위 표 +0.6과 같은 수준)
- 결론: 평평한 구간 규칙(잡음 수준 차이 -> 단순한 쪽)으로 **고정 w**, 전체 투구 모델은 수준 2 (2019 1등, 단독 점수도 더 좋음), 반감 50·k1 10·k2 10, w 0.2


## 5-2. 카운트 묶음별 투수 비율 (같음·유리·불리)

- 수준 2는 투수의 기대 대비 비율 r을 **모든 카운트에서 하나**로 셈 -> 투수마다 카운트 반응이 다르면 (예: 불리할 때만 직구를 리그보다 훨씬 많이 던짐) 0-0에 다른 카운트 성향이 섞임
- 카운트 묶음: 같음(볼 = 스트라이크: 0-0·1-1·2-2) / 투수 유리(스트라이크 > 볼) / 투수 불리(볼 > 스트라이크, 3-2 포함)
- 0-0 예측: 같음 묶음 비율 r_같음을 투수 전체 비율 r1 쪽으로 당김 (kg, 같음 묶음 공 수 단위) -> 투수×손 비율에 그 배수를 곱함
  - r_final = r2 × (r1g / r1), r1g = (O_g + kg·r1·ē_g) / (E_g + kg·ē_g) (O_g·E_g = 그 투수 같음 묶음 실제·기대 누적)
  - kg 크면 수준 2와 같음 (kg = ∞)
- 비교 (바꾸는 것 하나): 수준 2 vs 수준 2 + 같음 묶음, 둘 다 반감기(25·50·100)와 고정 w를 2019 겹마다 고름 (k1·k2 10 고정), 2020은 확인
- 주의: 투수×상황처럼 칸이 잘게 나뉘는 문제 (4-14) -> 같음 묶음은 모든 공의 약 46%라 표본은 큰 편

In [11]:
GRP = np.where(M_ALL["ball_before"] == M_ALL["strike_before"], "같음", np.where(M_ALL["strike_before"] > M_ALL["ball_before"], "유리", "불리"))
print(pd.Series(GRP).value_counts(normalize=True).round(3).to_dict())
for yr in Q:  # 연도별 데이터에 묶음 열 (같은 객체 유지 -> 캐시 그대로)
    M_E[yr]["grp"] = pd.Series(GRP, index=M_ALL.index).reindex(M_E[yr].index).to_numpy()
EVEN = {yr: M_E[yr][M_E[yr]["grp"] == "같음"] for yr in Q}  # 같음 묶음만 (연도별 객체 하나)

def level2g(q, year, half_life, k1, k2, kg, season_f=0.25):  # 수준 2 + 같음 묶음 비율 (0-0)
    d, de = M_E[year], EVEN[year]
    ebar = d.loc[d["GYEAR"] < year, CLASSES].mean().to_numpy()
    ebar_g = de.loc[de["GYEAR"] < year, CLASSES].mean().to_numpy()
    S = {}
    for nm, data, keys in [("1", d, ["pitcher_code"]), ("2", d, ["pitcher_code", "same_side"]), ("g", de, ["pitcher_code"])]:
        t = fc.prior_table(data, keys, CLASSES + E_COLS, half_life, season_f=season_f)
        s_, _ = fc.lookup(t, q, keys, CLASSES + E_COLS, season_f=season_f)
        S[nm] = (s_[:, :9], s_[:, 9:])
    r1 = (S["1"][0] + k1 * ebar) / (S["1"][1] + k1 * ebar)
    r2 = (S["2"][0] + k2 * r1 * ebar) / (S["2"][1] + k2 * ebar)
    r1g = (S["g"][0] + kg * r1 * ebar_g) / (S["g"][1] + kg * ebar_g)
    base = L[year].reindex(pd.MultiIndex.from_arrays([np.full(len(q), "0-0"), q["same_side"].to_numpy()])).to_numpy()
    P = base * r2 * (r1g / r1)
    return P / P.sum(1, keepdims=True)

HLG, KG = (25.0, 50.0, 100.0), (10.0, 30.0, 100.0, 300.0, 1000.0)
PH.update({hl: {yr: level2(Q[yr], yr, hl, 0.25, 10.0, 10.0) for yr in Q} for hl in HLG if hl not in PH})
PG = {(hl, kg): {yr: level2g(Q[yr], yr, hl, 10.0, 10.0, kg) for yr in Q} for hl in HLG for kg in KG}

def mix(yr, P_all, w):
    return (1 - w) * P13[yr] + w * P_all[yr]

rows, OUTG = [], {}
for nm, cands in {"수준 2": {(hl, None): PH[hl] for hl in HLG}, "수준 2 + 같음 묶음": PG}.items():
    LL = {(key, w): nll(mix(fc.VAL_YEAR, Pc, w), y19) for key, Pc in cands.items() for w in W}
    cv, best = cv_pick(LL)
    (key, w) = best
    d20 = b20 - nll(mix(fc.TEST_YEAR, cands[key], w), y20)
    OUTG[nm] = (cv, d20)
    lo, hi = game_ci(base19 - cv, g19)
    lo2, hi2 = game_ci(d20, g20)
    rows.append({"전체 투구 모델": nm, "후보 수": len(LL), "단독 2019 (1등 후보)": min(nll(Pc[fc.VAL_YEAR], y19).mean() for Pc in cands.values()),
                 "2019 교차 이득 ×1000": (base19 - cv).mean() * 1000, "95% 구간": f"{lo * 1000:+.1f} ~ {hi * 1000:+.1f}",
                 "2020 이득 ×1000": d20.mean() * 1000, "2020 95% 구간": f"{lo2 * 1000:+.1f} ~ {hi2 * 1000:+.1f}",
                 "2019 전체 1등": f"반감 {key[0]:g}" + ("" if key[1] is None else f", kg {key[1]:g}") + f", w {w:g}"})
display(pd.DataFrame(rows).set_index("전체 투구 모델").round(4))
d19 = OUTG["수준 2"][0] - OUTG["수준 2 + 같음 묶음"][0]
d20 = OUTG["수준 2 + 같음 묶음"][1] - OUTG["수준 2"][1]
lo, hi = game_ci(d19, g19); lo2, hi2 = game_ci(d20, g20)
print(f"같음 묶음 효과 (양수 = 좋아짐, ×1000): 2019 {d19.mean() * 1000:+.2f} (95% {lo * 1000:+.2f} ~ {hi * 1000:+.2f}), "
      f"2020 {d20.mean() * 1000:+.2f} (95% {lo2 * 1000:+.2f} ~ {hi2 * 1000:+.2f})")
display(pd.DataFrame({kg: [nll(PG[(hl, kg)][fc.VAL_YEAR], y19).mean() for hl in HLG] for kg in KG}, index=pd.Index(HLG, name="반감기"))
        .assign(**{"수준 2 (kg ∞)": [nll(PH[hl][fc.VAL_YEAR], y19).mean() for hl in HLG]}).round(5).rename_axis(columns="kg (단독 2019 log loss)"))

{'같음': 0.443, '유리': 0.286, '불리': 0.271}


,후보 수,단독 2019 (1등 후보),2019 교차 이득 ×1000,95% 구간,2020 이득 ×1000,2020 95% 구간,2019 전체 1등
전체 투구 모델,,,,,,,
수준 2,33,1.0943,4.2185,+1.8 ~ +6.7,3.6656,+1.8 ~ +5.5,"반감 50, w 0.2"
수준 2 + 같음 묶음,165,1.0915,4.1850,+1.8 ~ +6.6,3.6031,+1.8 ~ +5.4,"반감 50, kg 1000, w 0.2"


같음 묶음 효과 (양수 = 좋아짐, ×1000): 2019 -0.03 (95% -0.10 ~ +0.04), 2020 -0.06 (95% -0.12 ~ -0.01)


kg (단독 2019 log loss),10.0,30.0,100.0,300.0,1000.0,수준 2 (kg ∞)
반감기,,,,,,
25.0,1.10730,1.10443,1.10597,1.10871,1.11048,1.11146
50.0,1.09475,1.09229,1.09220,1.09412,1.09585,1.09697
100.0,1.09529,1.09297,1.09148,1.09197,1.09322,1.09433


#### 🔍 5-2 결과
#### 같음 묶음 비율은 전체 투구 모델 단독으로는 좋아지지만, 섞으면 효과 없음 (2019 −0.03, 2020 −0.06) -> 적용 안 함
- 단독 2019 초구 log loss: 수준 2 1.0943 (반감 100) -> 같음 묶음 1.0915 (반감 100·kg 100) -> 0-0과 같은 카운트(같음 묶음) 성향을 따로 두면 전체 투구 모델 자체는 좋아짐
- 섞은 2019 교차 이득: 수준 2 +4.22 vs 같음 묶음 +4.19, 2019 전체 1등 kg 1000 (= 거의 수준 2 그대로)
  - 같음 묶음 효과: 2019 −0.03 (95% −0.10 ~ +0.04), 2020 −0.06 (−0.12 ~ −0.01, 아주 작게 나빠짐)
- 이유: 같음 묶음(특히 0-0) 성향은 초구 모델이 이미 가진 정보 (B4가 그 투수 초구 기록으로 출발) -> 전체 투구 모델이 섞여서 주는 몫은 "카운트를 걸러 낸 투수의 일반 성향"이고, 0-0 쪽으로 다가갈수록 초구 모델과 겹침
  - 단독 점수가 좋아지는 방향과 섞어서 좋아지는 방향이 다를 수 있음 -> 판단은 계속 섞은 점수로
- 유리·불리 묶음은 0-0 예측에 직접 쓰이지 않음 (같음 묶음 비율만 0-0에 씀)


## 6. 저장

- `results/all_pitch_01/p00.parquet`: 2019·2020 초구의 전체 투구 모델 0-0 구종 확률 (수준 2 반감 50·k1 10·k2 10 = 5-1에서 고른 값) + B4 유효 초구 수
- `blend_params.csv`: 섞는 설정 (고정 w 0.2) -> 블렌딩 노트북(model_10 예정)에서 읽음
- compare는 `ADOPTED` 모델만 읽으므로 이 폴더는 들어가지 않음 (`per_pitch.parquet`도 아님)


In [12]:
LL = {(hl, w): nll(blend_h(fc.VAL_YEAR, hl, w, w, 1.0), y19) for hl in HL5 for w in W}
hl_b, w_b = min(LL, key=lambda k: LL[k].mean())  # 5-1 고정 w의 2019 전체 1등
out = pd.concat([pd.concat([Q[yr][fc.KEYS].reset_index(drop=True), pd.DataFrame(PH[hl_b][yr], columns=fc.P_TYPE),
                            pd.DataFrame({"b4_n": NB[yr]})], axis=1) for yr in Q], ignore_index=True)
os.makedirs("results/" + MODEL, exist_ok=True)
out.to_parquet(f"results/{MODEL}/p00.parquet", index=False)
pd.Series({"level": 2, "half_life": hl_b, "season_f": 0.25, "k1": 10.0, "k2": 10.0, "w": w_b}).to_csv(f"results/{MODEL}/blend_params.csv")
print("저장:", f"results/{MODEL}/p00.parquet", out.shape, f"/ 반감 {hl_b:g}, w {w_b:g}")


저장: results/all_pitch_01/p00.parquet (26073, 12) / 반감 50, w 0.2


## 다음 할 일 (초안 이후)

- [x] 카운트 반응이 투수마다 다른 몫 (같음 묶음 비율) -> 5-2, 단독은 좋아지지만 섞으면 효과 없음 (초구 모델과 겹침)
- 상황 효과: 0-0 리그 표 대신 초구 모델의 상황 보정을 그대로 쓰는 방법 (블렌딩이 아닌 곱하기) 검토
- [x] 블렌딩 비율을 투수 기록 수에 따라 (기록 적으면 w 크게) -> 5장, 반감기까지 교차 안에서 고르면 효과 없음 -> 고정 w
- 블렌딩 모델 (실전 위치까지 평가, compare) -> model_10
- 반감기 1등이 범위 끝이면 넓힘